# iPhone File Sorter

Sort files copied from your iPhone **Internal Storage** into folders by type:

- `Images/`
- `Videos/`
- `Documents/`
- `Excel/`
- `PDF/`
- `Other/`

## How to use
1. Copy iPhone folders from File Explorer (`This PC → Apple iPhone → Internal Storage`) to a local folder.
2. Edit the **Settings** cell below with your paths.
3. Run all cells (`Run → Run All Cells`).
4. Start with `DRY_RUN = True`, then set it to `False` to copy files.

## 1. Settings

Change these paths to match your laptop.

In [ ]:
from pathlib import Path

# Folder that contains the copied iPhone folders (202403_b, etc.)
SOURCE = Path(r"C:\Users\YourName\Documents\iPhone_Copy")

# Where sorted folders will be created
DESTINATION = Path(r"C:\Users\YourName\Documents\iPhone_Sorted")

# True = preview only (no files copied/moved)
DRY_RUN = True

# False = copy (safer). True = move files out of source.
MOVE = False

print("Source:", SOURCE)
print("Destination:", DESTINATION)
print("Dry run:", DRY_RUN)
print("Move:", MOVE)

## 2. Load sorter helpers

In [ ]:
import shutil
from collections import Counter

CATEGORY_EXTENSIONS = {
    "Images": {
        ".jpg", ".jpeg", ".png", ".heic", ".heif", ".gif",
        ".webp", ".bmp", ".tif", ".tiff", ".raw", ".dng",
    },
    "Videos": {
        ".mov", ".mp4", ".m4v", ".avi", ".mkv", ".3gp", ".mpg", ".mpeg",
    },
    "PDF": {".pdf"},
    "Excel": {".xls", ".xlsx", ".csv", ".xlsm", ".ods"},
    "Documents": {
        ".doc", ".docx", ".txt", ".rtf", ".pages", ".odt",
        ".ppt", ".pptx", ".key",
    },
}

EXTENSION_TO_CATEGORY = {
    ext: category
    for category, extensions in CATEGORY_EXTENSIONS.items()
    for ext in extensions
}

SKIP_NAMES = {".ds_store", "thumbs.db", "desktop.ini"}


def categorize(path: Path) -> str:
    return EXTENSION_TO_CATEGORY.get(path.suffix.lower(), "Other")


def unique_destination(dest_dir: Path, filename: str) -> Path:
    candidate = dest_dir / filename
    if not candidate.exists():
        return candidate
    stem = Path(filename).stem
    suffix = Path(filename).suffix
    counter = 1
    while True:
        candidate = dest_dir / f"{stem}_{counter}{suffix}"
        if not candidate.exists():
            return candidate
        counter += 1


def iter_source_files(source: Path):
    for path in source.rglob("*"):
        if path.is_file() and path.name.lower() not in SKIP_NAMES:
            yield path


def sort_files(source: Path, destination: Path, *, dry_run: bool = False, move: bool = False):
    if not source.exists() or not source.is_dir():
        raise FileNotFoundError(f"Source folder not found: {source}")

    counts: Counter[str] = Counter()
    action = "Moving" if move else "Copying"

    if not dry_run:
        destination.mkdir(parents=True, exist_ok=True)

    for src_file in iter_source_files(source):
        try:
            src_file.resolve().relative_to(destination.resolve())
            continue
        except (ValueError, OSError):
            pass

        category = categorize(src_file)
        category_dir = destination / category
        dest_file = unique_destination(category_dir, src_file.name)

        print(f"{action}: {src_file} -> {dest_file}")
        counts[category] += 1

        if dry_run:
            continue

        category_dir.mkdir(parents=True, exist_ok=True)
        if move:
            shutil.move(str(src_file), str(dest_file))
        else:
            shutil.copy2(str(src_file), str(dest_file))

    return dict(counts)


def print_summary(counts: dict, dry_run: bool) -> None:
    total = sum(counts.values())
    print("\n" + "=" * 40)
    print("DRY RUN — no files changed" if dry_run else "Done")
    print("=" * 40)
    if not counts:
        print("No files found.")
        return
    for category in ("Images", "Videos", "Documents", "Excel", "PDF", "Other"):
        if category in counts:
            print(f"  {category:12} {counts[category]}")
    print(f"  {'Total':12} {total}")

print("Helpers ready.")

## 3. Preview what will be sorted

Counts files in the source folder by type (no copying yet).

In [ ]:
source = SOURCE.expanduser().resolve()
destination = DESTINATION.expanduser().resolve()

if not source.exists():
    raise FileNotFoundError(
        f"Source not found: {source}\n"
        "Update SOURCE in the Settings cell to your copied iPhone folder."
    )

preview = Counter()
for f in iter_source_files(source):
    preview[categorize(f)] += 1

print(f"Scanning: {source}\n")
if not preview:
    print("No files found.")
else:
    for category, count in sorted(preview.items()):
        print(f"  {category:12} {count}")
    print(f"  {'Total':12} {sum(preview.values())}")

## 4. Run the sorter

With `DRY_RUN = True`, this only prints planned actions.

Set `DRY_RUN = False` in Settings, re-run Settings, then run this cell to copy files.

In [ ]:
if source == destination:
    raise ValueError("SOURCE and DESTINATION must be different folders.")

try:
    destination.relative_to(source)
    raise ValueError("DESTINATION cannot be inside SOURCE.")
except ValueError as exc:
    if "cannot be inside" in str(exc):
        raise

counts = sort_files(
    source,
    destination,
    dry_run=DRY_RUN,
    move=MOVE,
)
print_summary(counts, DRY_RUN)

if not DRY_RUN:
    print("\nOpen this folder to browse sorted files:")
    print(destination)

## Tips

- Keep `MOVE = False` unless you are sure you want files removed from the source folder.
- `.HEIC` photos may need **HEIF Image Extensions** from the Microsoft Store to preview in Windows.
- Chats (iMessage / WhatsApp) are usually not in these Internal Storage folders.